# Hack Apertus Track 2A — held-out submission

This notebook runs the checked-in competition pipeline on a Colab GPU.
Upload `see-it-say-it-pick-it-colab-heldout.zip` when prompted.
When the dataset picker opens, upload `hack-apertus-2a-heldout-inputs.zip`.
This notebook refreshes the Drive dataset cache so it cannot accidentally reuse
the public development dataset.
Input validation precedes model work. Apertus caches source programs,
then exits. Qwen/SAM and bounded reference recovery run in one process,
then exit. Final deterministic inference reads only caches and follows
source-bound parser -> complete resolver -> immutable obligation ledger ->
physical evidence -> centralized compiler -> grounding. Soft identity evidence
cannot authorize a plan. Production rules contain no benchmark-specific
instruction strings or object IDs; inference never reads ground truth.
Drive resumes only with matching fingerprints. Always upload the new source
ZIP, even when reusing Drive caches. Run all 14 cells in order without patch
cells: setup (2), project upload (3), Drive (4), dataset upload/reuse (5),
install/auth/config/build (6-9), Apertus and release (10), vision and release
(11-12), infer/validate (13), package (14). Cell 1 is this introduction.

Inventory semantic repair: at most one; targeted recovery: at most one
extra visual pass; Apertus parse: at most two attempts.
Complete plans stay in internal telemetry;
results retain the official first-pick submission schema. The final archive contains
`results_heldout.json`, ready for submission.

On failure, keep the Drive failure archive, disconnect/delete the
runtime, then run this notebook from the top with the new project ZIP.
Never rerun a model-stage cell while llama-server is alive. Hardware
fit, accuracy and end-to-end latency still require a live run.


In [ ]:
from collections import deque
from pathlib import Path, PurePosixPath
import shlex
import shutil
import subprocess
import importlib
import sys
import traceback
import csv
import hashlib
import json
import re
import stat
import tarfile
import tempfile
import time
from zipfile import ZipFile

content_root = Path("/content") if Path("/content").is_dir() else Path.cwd()
command_log_path = content_root / "see-pick-command.log"
persistent_root = None
stage = "setup"
DOWNLOAD_ARTIFACTS = True

def redact(value):
    if isinstance(value, dict):
        return {key: "[REDACTED]" if key.lower() in {
            "password", "authorization", "api_key", "api_token", "access_token", "tokens"
        } else redact(child) for key, child in value.items()}
    if isinstance(value, list):
        return [redact(child) for child in value]
    if isinstance(value, str):
        return re.sub(r"hf_[A-Za-z0-9]{8,}|sk-[A-Za-z0-9_-]{8,}|(?i:bearer)\s+[A-Za-z0-9._~+/=-]{8,}", "[REDACTED]", value)
    return value

def safe_copy(source, destination):
    if not source.is_file() or source.suffix not in {".json", ".yaml", ".log", ".txt"}:
        return
    text = source.read_text(encoding="utf-8", errors="replace")
    if source.suffix == ".json":
        try:
            text = json.dumps(redact(json.loads(text)), indent=2)
        except ValueError:
            text = redact(text)
    else:
        text = redact(text)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(text, encoding="utf-8")

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def save_llama_build_cache(source_bin, archive, identity):
    source_bin = Path(source_bin)
    archive = Path(archive)
    if not (source_bin / "llama-server").is_file():
        raise FileNotFoundError(source_bin / "llama-server")
    archive.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="llama-cache-save-", dir=content_root) as temp:
        temp = Path(temp)
        manifest = temp / "manifest.json"
        manifest.write_text(
            json.dumps({"version": 1, "identity": identity}, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        local_archive = temp / "llama-build.tar"
        with tarfile.open(local_archive, "w") as bundle:
            bundle.add(source_bin, arcname="bin")
            bundle.add(manifest, arcname="manifest.json")
        temporary = archive.with_suffix(archive.suffix + ".tmp")
        shutil.copy2(local_archive, temporary)
        temporary.replace(archive)
    print("Saved llama.cpp build cache:", archive)

def restore_llama_build_cache(archive, destination, identity):
    archive = Path(archive)
    destination = Path(destination)
    if not archive.is_file():
        return False
    try:
        with tempfile.TemporaryDirectory(
            prefix="llama-cache-restore-", dir=content_root
        ) as temp:
            temp = Path(temp)
            local_archive = temp / "llama-build.tar"
            shutil.copy2(archive, local_archive)
            extracted = temp / "extracted"
            extracted.mkdir()
            with tarfile.open(local_archive, "r") as bundle:
                bundle.extractall(extracted, filter="data")
            manifest = json.loads(
                (extracted / "manifest.json").read_text(encoding="utf-8")
            )
            restored = extracted / "bin"
            if manifest != {"version": 1, "identity": identity}:
                return False
            if not (restored / "llama-server").is_file():
                return False
            destination.parent.mkdir(parents=True, exist_ok=True)
            if destination.exists():
                shutil.rmtree(destination)
            shutil.move(restored, destination)
    except (OSError, ValueError, KeyError, json.JSONDecodeError, tarfile.TarError) as exc:
        print(f"Ignoring invalid llama.cpp build cache {archive}: {exc}")
        return False
    print("Restored llama.cpp build cache:", archive)
    return True

def select_dataset_archive(cache_root, upload, *, refresh=False):
    cache_root = Path(cache_root)
    archive = cache_root / "dataset-inputs.zip"
    manifest = cache_root / "dataset-inputs.json"
    if not refresh and (archive.exists() or manifest.exists()):
        if not archive.is_file() or not manifest.is_file():
            raise ValueError("Dataset cache is incomplete; refresh it explicitly.")
        expected = json.loads(manifest.read_text(encoding="utf-8"))["sha256"]
        if file_sha256(archive) != expected:
            raise ValueError("Dataset cache digest mismatch; refresh it explicitly.")
        print("Reusing Drive dataset inputs:", archive)
        return archive

    print("Upload hack-apertus-2a-heldout-inputs.zip.")
    uploaded = upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError("Upload exactly one dataset ZIP.")
    source = Path(zip_names[0])
    cache_root.mkdir(parents=True, exist_ok=True)
    digest = file_sha256(source)
    archive_tmp = archive.with_suffix(".zip.tmp")
    manifest_tmp = manifest.with_suffix(".json.tmp")
    shutil.copy2(source, archive_tmp)
    archive_tmp.replace(archive)
    manifest_tmp.write_text(json.dumps({"sha256": digest}, indent=2) + "\n",
                            encoding="utf-8")
    manifest_tmp.replace(manifest)
    print("Saved dataset inputs to Drive:", archive)
    return archive

def activate_project_source(project_root):
    source_root = (Path(project_root) / "src").resolve()
    if not (source_root / "see_pick" / "data.py").is_file():
        raise FileNotFoundError(f"Project source package is incomplete: {source_root}")
    source_text = str(source_root)
    sys.path[:] = [entry for entry in sys.path if entry != source_text]
    sys.path.insert(0, source_text)
    for name in tuple(sys.modules):
        if name == "see_pick" or name.startswith("see_pick."):
            sys.modules.pop(name, None)
    importlib.invalidate_caches()
    module = importlib.import_module("see_pick.data")
    module_path = Path(module.__file__).resolve()
    if not module_path.is_relative_to(source_root):
        raise ImportError(f"see_pick resolved outside uploaded project: {module_path}")
    print("Active project package:", module_path)
    return module

def save_progress(state, **details):
    payload = redact({"stage": stage, "state": state, "time_unix": time.time(), **details})
    if persistent_root is not None:
        persistent_root.mkdir(parents=True, exist_ok=True)
        destination = persistent_root / "progress.json"
        temporary = destination.with_suffix(".json.tmp")
        temporary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        temporary.replace(destination)
        safe_copy(command_log_path, persistent_root / "command.log")

ZIP_LIMITS = {
    "release": {"members": 2000, "file_bytes": 100 * 1024**2,
                "total_bytes": 1024**3, "compression_ratio": 200},
    "dataset": {"members": 100000, "file_bytes": 2 * 1024**3,
                "total_bytes": 8 * 1024**3, "compression_ratio": 500,
                "reserved_free_bytes": 20 * 1024**3},
}

def extract_zip(source, destination, *, profile="dataset"):
    destination = Path(destination).resolve()
    if destination.exists():
        raise FileExistsError(f"Extraction destination already exists: {destination}")
    limits = ZIP_LIMITS[profile]
    staging = Path(tempfile.mkdtemp(prefix=f".{destination.name}-extract-",
                                    dir=destination.parent)).resolve()
    try:
        with ZipFile(source) as archive:
            members = archive.infolist()
            if len(members) > limits["members"]:
                raise ValueError("ZIP member count exceeds extraction limit")
            seen = set(); total = 0
            for member in members:
                name = member.filename.rstrip("/"); path = PurePosixPath(name)
                kind = stat.S_IFMT(member.external_attr >> 16)
                if (not name or path.is_absolute() or ".." in path.parts
                    or "\\" in member.filename or path.as_posix() != name or name in seen):
                    raise ValueError(f"Unsafe ZIP member: {member.filename}")
                seen.add(name)
                if member.is_dir():
                    if kind not in {0, stat.S_IFDIR}:
                        raise ValueError(f"Special ZIP member: {name}")
                    continue
                if kind not in {0, stat.S_IFREG}:
                    raise ValueError(f"Special ZIP member: {name}")
                total += member.file_size
                ratio = member.file_size / max(member.compress_size, 1)
                if (member.file_size > limits["file_bytes"] or total > limits["total_bytes"]
                    or ratio > limits["compression_ratio"]):
                    raise ValueError(f"ZIP resource limit exceeded: {name}")
            reserve = limits.get("reserved_free_bytes", 0)
            if shutil.disk_usage(destination.parent).free < total + reserve:
                raise ValueError("ZIP extraction would violate free-space reserve")
            for member in members:
                name = member.filename.rstrip("/"); target = staging / name
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True); continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source_handle, target.open("xb") as target_handle:
                    shutil.copyfileobj(source_handle, target_handle, length=1024 * 1024)
        staging.replace(destination)
    except Exception:
        shutil.rmtree(staging, ignore_errors=True)
        raise

def verify_release_archive(source):
    with ZipFile(source) as archive:
        files = {}
        seen = set()
        for member in archive.infolist():
            name = member.filename.rstrip("/")
            path = PurePosixPath(name)
            if (not name or path.is_absolute() or ".." in path.parts
                or "\\" in name or path.as_posix() != name):
                raise ValueError(f"Unsafe release member: {member.filename}")
            if name in seen:
                raise ValueError(f"Release contains duplicate member: {name}")
            seen.add(name)
            kind = stat.S_IFMT(member.external_attr >> 16)
            if member.is_dir():
                if kind not in {0, stat.S_IFDIR}:
                    raise ValueError(f"Non-directory release member: {name}")
            elif kind in {0, stat.S_IFREG}:
                files[name] = member
            else:
                raise ValueError(f"Non-regular release member: {name}")
        manifests = [name for name in files if PurePosixPath(name).name == "RELEASE_MANIFEST.json"]
        if len(manifests) != 1:
            raise ValueError("Release must contain exactly one manifest")
        manifest_name = manifests[0]
        manifest = json.loads(archive.read(files[manifest_name]))
        hashes = manifest.get("sha256")
        if not isinstance(hashes, dict) or not {"SOURCE_COMMIT.txt", "pyproject.toml"} <= hashes.keys():
            raise ValueError("Release manifest is missing required files")
        prefix = str(PurePosixPath(manifest_name).parent)
        prefix = "" if prefix == "." else prefix + "/"
        expected = {prefix + name for name in hashes} | {manifest_name}
        if set(files) != expected:
            raise ValueError("Release regular-file coverage does not match manifest")
        for relative, digest in hashes.items():
            if not isinstance(digest, str) or re.fullmatch(r"[0-9a-f]{64}", digest) is None:
                raise ValueError(f"Invalid release digest: {relative}")
            if hashlib.sha256(archive.read(files[prefix + relative])).hexdigest() != digest:
                raise ValueError(f"Release manifest mismatch: {relative}")
        commit = archive.read(files[prefix + "SOURCE_COMMIT.txt"]).decode("utf-8").strip()
        if manifest.get("source_commit") != commit:
            raise ValueError("Release source commit mismatch")
        return manifest

def write_run_manifest(run_dir):
    destination = run_dir / "RUN_MANIFEST.json"
    temporary = run_dir / "RUN_MANIFEST.json.tmp"
    if not (run_dir / "results.json").is_file():
        raise FileNotFoundError("Validated results.json is required before run manifest")
    hashes = {path.relative_to(run_dir).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
              for path in sorted(run_dir.rglob("*"))
              if path.is_file() and path not in {destination, temporary}}
    temporary.write_text(json.dumps({"version": 1, "sha256": hashes}, indent=2, sort_keys=True)
                         + "\n", encoding="utf-8")
    temporary.replace(destination)
    return destination

def runtime_manifest(project_root, config, cache_fingerprints):
    # Every runtime source participates, including parser prompts,
    # schemas, normalization, resolver, compiler and grounding contracts.
    runtime_paths = [
        *sorted((project_root / "src/see_pick").rglob("*.py")),
        project_root / "starter/scene_io.py",
        project_root / "schema/result.schema.json",
        project_root / "pyproject.toml",
        *sorted((project_root / "configs").rglob("*.yaml")),
    ]
    source_sha256 = {
        path.relative_to(project_root).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
        for path in runtime_paths if path.is_file()
    }
    manifest = {"config": config, "cache_fingerprints": cache_fingerprints,
                "source_sha256": source_sha256,
                "bounds": {"max_parse_attempts": 2,
                           "max_inventory_attempts": 2, "max_recovery_passes": 1}}
    encoded = json.dumps(manifest, sort_keys=True, separators=(",", ":")).encode()
    return {**manifest, "fingerprint": hashlib.sha256(encoded).hexdigest()}

def validate_dataset(root):
    import numpy as np
    from see_pick.data import load_challenge
    root = Path(root).resolve()
    with (root / "instructions.csv").open(encoding="utf-8-sig", newline="") as handle:
        for row in csv.DictReader(handle):
            if not (root / (row.get("scene") or "")).resolve().is_relative_to(root):
                raise ValueError("Unsafe dataset scene path")
    scenes, instructions = load_challenge(root)
    ids = [item.instruction_id for item in instructions]
    if not scenes or len(set(ids)) != len(ids):
        raise ValueError("Dataset has no scenes or duplicate instruction IDs")
    for scene in scenes:
        matrix = np.asarray(scene.calibration["K"])
        if matrix.shape != (3, 3) or not np.isfinite(matrix).all() or np.linalg.det(matrix) == 0:
            raise ValueError("Invalid camera calibration")
        transform = scene.calibration["base_T_camera"]
        if transform is not None and (np.shape(transform) != (4, 4) or not np.isfinite(transform).all()):
            raise ValueError("Invalid camera transform")
        if scene.rgb.ndim != 3 or scene.rgb.shape[2] != 3 or not scene.rgb.size:
            raise ValueError("Invalid scene RGB asset")
        if scene.depth_mm is not None and scene.depth_mm.shape != scene.rgb.shape[:2]:
            raise ValueError("Invalid scene depth dimensions")
        if scene.xyz is not None and scene.xyz.shape != scene.rgb.shape:
            raise ValueError("Invalid scene XYZ dimensions")
        if not np.isfinite(scene.gripper_max_opening_mm) or scene.gripper_max_opening_mm <= 0:
            raise ValueError("Invalid gripper calibration")
    print(f"Dataset validated: {len(scenes)} scenes, {len(instructions)} instructions")
    return scenes, instructions

def verify_intent_cache(root, config, instructions):
    from see_pick.cli import _intent_cache
    cache = _intent_cache(config, root)
    cache.require_executable(item.text for item in instructions)

def verify_staged_caches(root, config, scenes, instructions, intent_root):
    from see_pick.cli import _scene_programs, _recovery_layers
    from see_pick.vision_cache import VisionCacheBackend
    from see_pick.vision_cache import image_cache_key
    from see_pick.vision_provenance import vision_pipeline_fingerprint
    cache = VisionCacheBackend(root, pipeline_fingerprint=vision_pipeline_fingerprint(config))
    if any(cache.probe(scene.rgb, scene=scene).status != "valid" for scene in scenes):
        raise RuntimeError("Vision cache incomplete; restart and resume vision stage")
    verify_intent_cache(intent_root, config, instructions)
    grouped = _scene_programs(config, instructions, intent_root)
    by_image = {image_cache_key(scene.rgb): grouped.get(scene.scene_id, ()) for scene in scenes}
    layers = _recovery_layers(config, root, intent_root, by_image)
    if any(layers[image_cache_key(scene.rgb)].probe(scene.rgb).status != "valid" for scene in scenes):
        raise RuntimeError("Recovery cache incomplete; restart and resume vision stage")
    return {key: layer.fingerprint for key, layer in sorted(layers.items())}

def check_gpu(label):
    # Query without importing torch in the notebook kernel: only the
    # short-lived vision subprocess owns Qwen/SAM CUDA allocations.
    run_command(["nvidia-smi"])
    output = subprocess.check_output([
        "nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free",
        "--format=csv,noheader,nounits"], text=True)
    print(f"{label}: name,total/used/free MiB: {output.strip()}", flush=True)
    save_progress("gpu_check", gpu=output.strip())
    if label in {"vision_released", "apertus_released"}:
        owners = subprocess.check_output([
            "nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader,nounits"], text=True)
        if owners.strip():
            raise RuntimeError(f"GPU process remains at {label}; save diagnostics and restart runtime")
    return int(output.splitlines()[0].split(",")[1].strip()) / 1024

def package_failure():
    project_root = globals().get("repo", content_root / "see-pick-work")
    staging = content_root / "hack-apertus-2a-failure"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)
    candidates = [
        (command_log_path, staging / "command.log"),
        (
            project_root / "SOURCE_COMMIT.txt",
            staging / "SOURCE_COMMIT.txt",
        ),
        (
            Path(globals().get("config_path", project_root / "configs/models.colab.yaml")),
            staging / "models.colab.yaml",
        ),
        (globals().get("vision_diagnostics", project_root / "artifacts/vision-diagnostics") / "runtime-manifest.json",
         staging / "runtime-manifest.json"),
        (
            project_root / "artifacts/colab-llama-server.log",
            staging / "colab-llama-server.log",
        ),
    ]
    for source, destination in candidates:
        if source.is_file():
            safe_copy(source, destination)
    diagnostics = Path(
        globals().get(
            "vision_diagnostics",
            project_root / "artifacts/vision-diagnostics",
        )
    )
    if diagnostics.is_dir():
        for source in diagnostics.rglob("*.json"):
            safe_copy(source, staging / "vision-diagnostics" / source.relative_to(diagnostics))
    else:
        print(f"No vision diagnostics found at {diagnostics}", flush=True)
    intent_root = globals().get("intent_cache")
    if intent_root is not None:
        for source in (Path(intent_root) / "intents").glob("*.json"):
            safe_copy(source, staging / "intent-diagnostics" / source.name)
    if "runtime_contract" in globals():
        (staging / "runtime-manifest.json").write_text(
            json.dumps(redact(runtime_contract), indent=2), encoding="utf-8")
    active_run = globals().get("run_dir")
    if active_run is not None:
        for source in Path(active_run).glob("*.json"):
            safe_copy(source, staging / "run-diagnostics" / source.name)
    archive = shutil.make_archive(
        str(content_root / "hack-apertus-2a-failure"),
        "zip",
        root_dir=staging,
    )
    if persistent_root is not None:
        persistent_root.mkdir(parents=True, exist_ok=True)
        shutil.copy2(archive, persistent_root / Path(archive).name)
    save_progress("failed", archive=str(archive))
    try:
        from google.colab import files
        if DOWNLOAD_ARTIFACTS:
            files.download(archive)
    except Exception as exc:
        print(f"Failure archive is available at {archive}: {exc}", flush=True)
    return archive

def run_command(command):
    command = [str(part) for part in command]
    display = shlex.join(command)
    display = redact(display)
    print(f"\n$ {display}", flush=True)
    save_progress("running", command=display)
    process = run_python_step(f"Starting {display}", lambda: subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    ))
    tail = deque(maxlen=120)
    assert process.stdout is not None
    with command_log_path.open("a", encoding="utf-8") as command_log:
        command_log.write(f"\n$ {display}\n")
        for line in process.stdout:
            line = redact(line)
            print(line, end="", flush=True)
            command_log.write(line)
            command_log.flush()
            tail.append(line)
    return_code = process.wait()
    if return_code:
        print(
            f"\n--- Command failed with exit code {return_code}: {display} ---",
            flush=True,
        )
        if tail:
            print("Last output lines (repeated for diagnosis):", flush=True)
            print("".join(tail), flush=True)
        try:
            package_failure()
        except Exception as exc:
            print(f"Could not package failure diagnostics: {exc}", flush=True)
        raise RuntimeError(
            f"Command failed with exit code {return_code}: {display}"
        )
    save_progress("command_complete", command=display)

def run_python_step(label, action):
    print(f"\n{label}", flush=True)
    try:
        return action()
    except Exception:
        traceback.print_exc()
        with command_log_path.open("a", encoding="utf-8") as command_log:
            command_log.write(redact(traceback.format_exc()))
        try:
            package_failure()
        except Exception as exc:
            print(f"Could not package failure diagnostics: {exc}", flush=True)
        raise


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
import shutil

from google.colab import files

print("Upload see-it-say-it-pick-it-colab.zip")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise RuntimeError("Upload exactly one project ZIP.")

workspace = Path("/content/see-pick-work")
if workspace.exists():
    raise RuntimeError(f"{workspace} already exists; restart the runtime for a clean run.")
manifest = run_python_step("Verify exact project ZIP coverage", lambda: verify_release_archive(zip_names[0]))
extract_zip(zip_names[0], workspace, profile="release")
candidates = list(workspace.rglob("pyproject.toml"))
if len(candidates) != 1:
    raise RuntimeError(f"Expected one project root, found {len(candidates)}")
repo = candidates[0].parent
source_commit = repo / "SOURCE_COMMIT.txt"
if not source_commit.is_file():
    raise FileNotFoundError("Project ZIP is missing SOURCE_COMMIT.txt")
os.chdir(repo)
print("Project:", repo)
print("Source commit:", source_commit.read_text(encoding="utf-8").strip())

actual_files = {path.relative_to(repo).as_posix() for path in repo.rglob("*") if path.is_file()}
if actual_files != set(manifest["sha256"]) | {"RELEASE_MANIFEST.json"}:
    raise ValueError("Extracted project coverage does not match manifest")
if manifest["source_commit"] != source_commit.read_text(encoding="utf-8").strip():
    raise ValueError("Release source commit mismatch")
for relative, digest in manifest["sha256"].items():
    target = (repo / relative).resolve()
    if not target.is_relative_to(repo.resolve()) or hashlib.sha256(target.read_bytes()).hexdigest() != digest:
        raise ValueError(f"Release manifest mismatch: {relative}")


In [ ]:
DRIVE_DIR = "hack-apertus-2a"

from google.colab import drive
drive.mount("/content/drive")
persistent_root = Path("/content/drive/MyDrive") / DRIVE_DIR
persistent_root.mkdir(parents=True, exist_ok=True)
print("Persistent artifacts:", persistent_root)
save_progress("mounted")


In [ ]:
REFRESH_DATASET_INPUTS = True

dataset_archive = select_dataset_archive(
    persistent_root / "inputs", files.upload, refresh=REFRESH_DATASET_INPUTS)
dataset_workspace = content_root / "see-pick-dataset"
if dataset_workspace.exists():
    raise RuntimeError("Dataset workspace exists; delete/restart runtime before retrying.")
extract_zip(dataset_archive, dataset_workspace, profile="dataset")
instruction_files = list(dataset_workspace.rglob("instructions.csv"))
if len(instruction_files) != 1:
    raise ValueError("Dataset ZIP must contain exactly one instructions.csv")
dataset = instruction_files[0].parent
if not (dataset / "calibration.json").is_file():
    raise FileNotFoundError("Dataset ZIP is missing calibration.json")


In [ ]:
import sys

run_command([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"])
run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==4.57.1",
        "accelerate==1.10.1",
        "qwen-vl-utils==0.0.14",
        "bitsandbytes==0.48.2",
        "huggingface-hub==0.36.0",
        "lm-format-enforcer==0.11.3",
    ]
)
activate_project_source(repo)
scenes, instructions = run_python_step("Validate all inputs before model work", lambda: validate_dataset(dataset))
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select Runtime > Change runtime type > GPU, then reconnect.")
total_memory_gib = run_python_step("GPU memory before models", lambda: check_gpu("before_models"))


In [ ]:
from huggingface_hub import login, notebook_login

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
if token:
    login(token=token, add_to_git_credential=False)
else:
    notebook_login()


In [ ]:
import yaml

config_path = Path("configs/models.colab.yaml")
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
low_memory_gpu = total_memory_gib < 30
print(f"GPU total_memory={total_memory_gib:.1f} GiB; fit must be measured live")
config["vision"]["inventory"]["load_in_4bit"] = low_memory_gpu
config["vision"]["inventory"]["max_pixels"] = 802816
config["vision"]["set_of_mark"]["enabled"] = False
config["perception_recovery"]["enabled"] = True
config_path = Path("artifacts/models.colab.yaml")
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

artifact_root = persistent_root or Path("artifacts")
from see_pick.cli import _intent_cache
from see_pick.vision_provenance import vision_pipeline_fingerprint
vision_fingerprint = vision_pipeline_fingerprint(config)
vision_cache = artifact_root / "colab-vision-cache-v4" / vision_fingerprint.split(":")[-1]
intent_fingerprint = _intent_cache(config, artifact_root).fingerprint
intent_cache = artifact_root / "colab-intent-cache" / intent_fingerprint
recovery_fingerprints = {}
runtime_contract = runtime_manifest(repo, config, {"base_vision": vision_fingerprint,
    "intent": intent_fingerprint, "recovery": recovery_fingerprints})
vision_diagnostics = artifact_root / "vision-diagnostics" / runtime_contract["fingerprint"]
vision_diagnostics.mkdir(parents=True, exist_ok=True)
(vision_diagnostics / "runtime-manifest.json").write_text(json.dumps(runtime_contract, indent=2), encoding="utf-8")
print("Runtime fingerprint:", runtime_contract["fingerprint"])
print("Retry bounds:", runtime_contract["bounds"])
print("Visual cache:", vision_cache, "Intent cache:", intent_cache)
stage = "configured"


In [ ]:
llama_root = Path("/content/llama.cpp-b11490")
llama_revision = "b11490"
import platform

nvcc_output = subprocess.check_output(["nvcc", "--version"], text=True)
cuda_match = re.search(r"release\s+(\d+\.\d+)", nvcc_output)
if cuda_match is None:
    raise RuntimeError("Could not identify the CUDA compiler version")
compute_capability = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
    text=True,
).splitlines()[0].strip()
cuda_architecture = compute_capability.replace(".", "")
build_identity = "-".join(
    [
        llama_revision,
        platform.system().lower(),
        platform.machine().lower(),
        f"cuda{cuda_match.group(1)}",
        f"sm{cuda_architecture}",
    ]
)
llama_bin = llama_root / "build/bin"
llama_server = llama_bin / "llama-server"
build_cache = persistent_root / "tool-cache" / f"llama.cpp-{build_identity}.tar"

def llama_server_works(path):
    try:
        result = subprocess.run(
            [str(path), "--version"],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=30,
        )
    except (OSError, subprocess.TimeoutExpired):
        return False
    return result.returncode == 0

build_ready = llama_server_works(llama_server)
if not build_ready:
    shutil.rmtree(llama_root, ignore_errors=True)
    build_ready = restore_llama_build_cache(
        build_cache, llama_bin, build_identity
    ) and llama_server_works(llama_server)
if not build_ready:
    shutil.rmtree(llama_root, ignore_errors=True)
    run_command(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            llama_revision,
            "https://github.com/ggml-org/llama.cpp.git",
            str(llama_root),
        ]
    )
    run_command(
        [
            "cmake",
            "-S",
            str(llama_root),
            "-B",
            str(llama_root / "build"),
            "-DGGML_CUDA=ON",
            "-DLLAMA_CURL=OFF",
            "-DCMAKE_BUILD_TYPE=Release",
            f"-DCMAKE_CUDA_ARCHITECTURES={cuda_architecture}",
        ]
    )
    run_command(
        [
            "cmake",
            "--build",
            str(llama_root / "build"),
            "--config",
            "Release",
            "-j",
            "2",
            "--target",
            "llama-server",
        ]
    )
    run_command([llama_server, "--version"])
    save_llama_build_cache(llama_bin, build_cache, build_identity)
print("llama.cpp build ready:", build_identity)


In [ ]:
if stage != "configured":
    raise RuntimeError("Configure and validate inputs before starting Apertus")
import time
import requests
from huggingface_hub import hf_hub_download

runtime_artifact = config["apertus"]["runtime_artifact"]
GGUF_REPO = runtime_artifact["repo_id"]
GGUF_REVISION = runtime_artifact["revision"]
GGUF_FILE = runtime_artifact["filename"]
stage = "apertus_start"
run_python_step("GPU memory before Apertus", lambda: check_gpu("before_apertus"))
gguf = run_python_step(
    "Downloading pinned Apertus GGUF.",
    lambda: hf_hub_download(
        GGUF_REPO,
        GGUF_FILE,
        revision=GGUF_REVISION,
    ),
)

server_log_path = Path("artifacts/colab-llama-server.log")
server_log_path.parent.mkdir(parents=True, exist_ok=True)
server_log = server_log_path.open("w", encoding="utf-8")
server = run_python_step("Start Apertus", lambda: subprocess.Popen(
    [
        str(llama_server),
        "--model",
        str(gguf),
        "--alias",
        "swiss-ai/Apertus-v1.5-8B",
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
        "--ctx-size",
        "8192",
        "--n-gpu-layers",
        "99",
    ],
    stdout=server_log,
    stderr=subprocess.STDOUT,
))
try:
    for _ in range(180):
        if server.poll() is not None:
            package_failure()
            raise RuntimeError(f"llama-server exited; inspect {server_log_path}")
        try:
            response = requests.get("http://127.0.0.1:8000/v1/models", timeout=2)
            if response.ok:
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    else:
        package_failure()
        raise TimeoutError(f"llama-server did not become ready; inspect {server_log_path}")
    print("Apertus server ready.")

    stage = "cache_intents"
    run_command([sys.executable, "-m", "see_pick.cli", "cache-intents",
        "--dataset", str(dataset), "--output", str(intent_cache),
        "--config", str(config_path)])
    run_python_step("Verify every intent entry", lambda: verify_intent_cache(intent_cache, config, instructions))
finally:
    try:
        server.terminate()
        try:
            server.wait(timeout=30)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait(timeout=30)
    finally:
        server_log.close()
run_python_step("Apertus GPU release gate", lambda: check_gpu("apertus_released"))
stage = "apertus_released"
save_progress("complete")


In [ ]:
from huggingface_hub import snapshot_download

QWEN_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
QWEN_REVISION = "cc594898137f460bfe9f0759e9844b3ce807cfb5"
SAM_ID = "facebook/sam2.1-hiera-large"
SAM_REVISION = "665f8e2ad61cf5f53d65644ff27c8ee525124610"

def download_vision_snapshots():
    snapshot_download(QWEN_ID, revision=QWEN_REVISION)
    snapshot_download(SAM_ID, revision=SAM_REVISION)
    settings = config["perception_recovery"]
    snapshot_download(settings["model_id"], revision=settings["revision"])

run_python_step("Downloading pinned Qwen and SAM snapshots.", download_vision_snapshots)
print("Pinned vision snapshots are ready.")


In [ ]:
if stage != "apertus_released":
    raise RuntimeError("Release Apertus before starting vision")
stage = "cache_vision_recovery"
cache_command = [sys.executable, "-m", "see_pick.cli", "cache-vision",
    "--dataset", str(dataset), "--output", str(vision_cache),
    "--config", str(config_path), "--diagnostics", str(vision_diagnostics),
    "--intent-cache", str(intent_cache)]
run_command(cache_command)
recovery_fingerprints = run_python_step("Verify base and recovery caches",
    lambda: verify_staged_caches(vision_cache, config, scenes, instructions, intent_cache))
run_python_step("Vision GPU release gate", lambda: check_gpu("vision_released"))
runtime_contract = runtime_manifest(repo, config, {"base_vision": vision_fingerprint,
    "intent": intent_fingerprint, "recovery": recovery_fingerprints})
(vision_diagnostics / "runtime-manifest.json").write_text(
    json.dumps(runtime_contract, indent=2), encoding="utf-8")
stage = "vision_released"
save_progress("complete", runtime_fingerprint=runtime_contract["fingerprint"])


In [ ]:
from datetime import datetime, timezone

TEAM_NAME = "Nightcats"
if stage != "vision_released":
    raise RuntimeError("Release vision before deterministic inference")
stage = "deterministic_infer"

run_root = (persistent_root / "runs") if persistent_root else Path("artifacts/public")
run_dir = run_root / (
    "heldout-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
)
run_dir.mkdir(parents=True)
shutil.copy2(config_path, run_dir / "models.yaml")
shutil.copy2(source_commit, run_dir / "git-sha.txt")
(run_dir / "runtime-manifest.json").write_text(json.dumps(runtime_contract, indent=2), encoding="utf-8")
results = run_dir / "results.json"
telemetry = run_dir / "telemetry.json"

run_command(
    [
        sys.executable,
        "-m",
        "see_pick.cli",
        "infer",
        "--dataset",
        str(dataset),
        "--output",
        str(results),
        "--team",
        TEAM_NAME,
        "--config",
        str(config_path),
        "--telemetry",
        str(telemetry),
        "--vision-cache",
        str(vision_cache),
        "--intent-cache",
        str(intent_cache),
    ]
)
run_command(
    [
        sys.executable,
        "-m",
        "see_pick.cli",
        "validate",
        "--results",
        str(results),
        "--dataset",
        str(dataset),
    ]
)
audit_path = vision_diagnostics / "vision-cache-audit.json"
if audit_path.is_file():
    shutil.copy2(audit_path, run_dir / audit_path.name)
heldout_results = run_dir / "results_heldout.json"
shutil.copy2(results, heldout_results)
run_python_step("Persist validated run artifact hashes", lambda: write_run_manifest(run_dir))
print("Held-out submission file:", heldout_results)
print("Inference complete:", run_dir)
save_progress("validated", run_dir=str(run_dir))


In [ ]:
archive = shutil.make_archive(
    "/content/hack-apertus-2a-heldout-results",
    "zip",
    root_dir=run_dir,
)
if persistent_root is not None:
    shutil.copy2(archive, persistent_root / Path(archive).name)
save_progress("complete", archive=str(archive), run_dir=str(run_dir))
if DOWNLOAD_ARTIFACTS:
    files.download(archive)
